In [3]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

In [5]:
from selenium.common.exceptions import TimeoutException

#  start selenium webdriver
driver = webdriver.Chrome()
driver.set_page_load_timeout(30)
try:
    url = "https://www.daraz.pk/catalog/?q=wireless+earbuds"

    driver.get(url)
    time.sleep(3)  # wait for 3 seconds to load the page
    
    #save the page source to a file
    html_content = driver.page_source
    with open("raw.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    print("Page source saved to raw.html")
# Add this to the import cell (or before the try block):

# Replace the placeholder with:
except TimeoutException:
    print("Page load timed out. Please check your internet connection or the website's availability.")
finally:
    html_content = driver.page_source
with open("raw.html", "w", encoding="utf-8") as f:
    f.write(html_content)
    driver.quit()        

Page source saved to raw.html


In [4]:
# Open the saved raw HTML for the Daraz search results page
with open('raw.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f, "html.parser")

product_links =[]
for link in soup.find_all("a",href=True):
    href=link.get("href")
    if href and "/products" in href:
        product_links.append(href)
    
#
# Keep only valid Daraz product URLs and remove any empty or malformed entries before deduplicating them

product_links =list(dict.fromkeys(product_links))
    
# Print the total number of unique product links found on the page
print(f'Total Unique Products Links: {len(product_links)}')


Total Unique Products Links: 40


In [5]:
product_links

['//www.daraz.pk/products/tws-wireless-earbuds-with-led-display-enc-noise-cancellation-hifi-stereo-bluetooth-53-ear-buds-woth-mic-i1965132501.html',
 '//www.daraz.pk/products/-i1954493616.html',
 '//www.daraz.pk/products/smart-tws-airbuds-bluetooth-50-wireless-earphones-with-ai-voice-translator-touch-control-charging-box-i1965246033.html',
 '//www.daraz.pk/products/12-i894764349.html',
 '//www.daraz.pk/products/m10-m90-tws-eir-pods-51-9d-hifi-2000mah-i433557338.html',
 '//www.daraz.pk/products/led-i1964559878.html',
 '//www.daraz.pk/products/m04-stereo-earbuds-tws-wireless-earbuds-with-led-display-enc-noise-cancellation-hifi-stereo-bluetooth-53-available-in-black-white-i948285745.html',
 '//www.daraz.pk/products/m10-tws-3500mah-9d-i1964237526.html',
 '//www.daraz.pk/products/-i1947434490.html',
 '//www.daraz.pk/products/31-52-7-i280841902.html',
 '//www.daraz.pk/products/-i1964603364.html',
 '//www.daraz.pk/products/2nd-i1948689808.html',
 '//www.daraz.pk/products/-i1964042544.html',
 

In [16]:
import time
import pandas as  pd
from selenium import webdriver
try:
    driver = webdriver.Chrome()

    products =[]

    #  test first 10 products

    for i,url in enumerate(product_links[:10],start=1):
        if url.startswith("//"):
            url= "https:"+url
            
            
        driver.get(url)
        time.sleep(2) #

        product_soup = BeautifulSoup(driver.page_source,'html.parser')

        # name
        h1 = product_soup.find('h1')
        # product name
        product_name = h1.get_text(strip=True) if h1 else None

        # add data into dictionary farmet in list

        products.append(
            {
                "Product_Name":product_name,
                "Product_url":url
            }
        )
        
        print(f'{i} - Scrapped:  {product_name}')
        
except Exception as e:
        print(f"{i} - Failed to scrape {url}: {e}")


1 - Scrapped:  🔥 TWS Wireless Earbuds with LED Display – ENC Noise Cancellation, HiFi Stereo, Bluetooth 5.3 Ear Buds |  woth Mic
2 - Scrapped:  Lanteso True Wireless Earbuds, Bluetooth Earbuds with Mics Clear Call Touch Control Bluetooth Headphones with Bass Sound in Ear Earphones for Music,Home Office
3 - Scrapped:  Smart TWS Airbuds – Bluetooth 5.0 Wireless Earphones with AI Voice Translator, Touch Control, Charging Box
4 - Scrapped:  Air-podTWS 12 black Mini Bluetooth Wireless Headset_(True Wireless Stereo) Bluetooth Earbuds Bluetooth Handsfree  Dual Sound Bass, Charging Case and Extra Ear-Buds, Pop-Up Feature Compatible with All Devices
5 - Scrapped:  Original M10 & M90 TWS  EIR PODS Wireless Earphones Touch Control 5.1 with power bank charger Headset Waterproof 9D Hifi Quality Earbuds 2000mAh Stereo Sports Headphones With Microphone
6 - Scrapped:  Wireless Earbuds Headphones Bluetooth Earpods Sports Stereo Airbuds Noise Canceling Headphone Microphone Speaker with LED light headpho

AttributeError: module 'pandas' has no attribute 'head'

In [22]:
df = pd.DataFrame(products)
df.to_csv("products.csv", index=False, encoding="utf-8-sig")
print("DataFrame saved to products.csv")

DataFrame saved to products.csv


In [26]:
import re
import json
import pandas as pd
import time

products = []

for i, url in enumerate(product_links, start=1):

    if url.startswith("//"):
        url = "https:" + url

    driver.get(url)
    time.sleep(2)

    html = driver.page_source

    # Find Daraz product tracking data
    match = re.search(
        r'var pdpTrackingData = "(.*?)";',
        html,
        re.DOTALL
    )

    if match:
        try:
            data_string = match.group(1)
            data_string = data_string.replace('\\"', '"')

            data = json.loads(data_string)

            product = {
                "Product_Name": data.get("pdt_name"),
                "Price": data.get("pdt_price"),
                "Brand": data.get("brand_name"),
                "SKU": data.get("pdt_sku"),
                "Category": data.get("pdt_category"),
                "Seller": data.get("seller_name"),
                "Product_URL": url
            }

            products.append(product)

            print(f"{i}/40 - {product['Product_Name']}")

        except Exception as e:
            print(f"{i}/40 - JSON Error:", e)

    else:
        print(f"{i}/40 - Tracking data not found")


print("\nScraping completed!")
print("Products collected:", len(products))

1/40 - 🔥 TWS Wireless Earbuds with LED Display – ENC Noise Cancellation, HiFi Stereo, Bluetooth 5.3 Ear Buds |  woth Mic
2/40 - Lanteso True Wireless Earbuds, Bluetooth Earbuds with Mics Clear Call Touch Control Bluetooth Headphones with Bass Sound in Ear Earphones for Music,Home Office
3/40 - Smart TWS Airbuds – Bluetooth 5.0 Wireless Earphones with AI Voice Translator, Touch Control, Charging Box
4/40 - Air-podTWS 12 black Mini Bluetooth Wireless Headset_(True Wireless Stereo) Bluetooth Earbuds Bluetooth Handsfree  Dual Sound Bass, Charging Case and Extra Ear-Buds, Pop-Up Feature Compatible with All Devices
5/40 - Original M10 & M90 TWS  EIR PODS Wireless Earphones Touch Control 5.1 with power bank charger Headset Waterproof 9D Hifi Quality Earbuds 2000mAh Stereo Sports Headphones With Microphone
6/40 - Wireless Earbuds Headphones Bluetooth Earpods Sports Stereo Airbuds Noise Canceling Headphone Microphone Speaker with LED light headphones
7/40 - M04 Stereo Earbuds TWS Wireless Earbu

In [34]:
df = pd.DataFrame(products)
df.head()
#df.to_csv(r'EarbudsDataFrame.csv',index= False,encoding='utf-8')

,Product_Name,Price,Brand,SKU,Category,Seller,Product_URL
0,🔥 TWS Wireless Earbuds with LED Display – ENC ...,"Rs. 3,500",No Brand,1965132501,"[TV, Audio / Video, Gaming & Wearables, Audio,...",,https://www.daraz.pk/products/tws-wireless-ear...
1,"Lanteso True Wireless Earbuds, Bluetooth Earbu...","Rs. 1,999",No Brand,1954493616,"[TV, Audio / Video, Gaming & Wearables, Audio,...",,https://www.daraz.pk/products/-i1954493616.html
2,Smart TWS Airbuds – Bluetooth 5.0 Wireless Ear...,"Rs. 1,999",No Brand,1965246033,"[TV, Audio / Video, Gaming & Wearables, Audio,...",,https://www.daraz.pk/products/smart-tws-airbud...
3,Air-podTWS 12 black Mini Bluetooth Wireless He...,"Rs. 4,999",No Brand,894764349,"[TV, Audio / Video, Gaming & Wearables, Audio,...",,https://www.daraz.pk/products/12-i894764349.html
4,Original M10 & M90 TWS EIR PODS Wireless Earp...,"Rs. 3,500",No Brand,433557338,"[TV, Audio / Video, Gaming & Wearables, Audio,...",,https://www.daraz.pk/products/m10-m90-tws-eir-...


In [31]:
print(f"Total Collected {len(df)}")
print(f"Unique {df.Product_Name.unique()}")

Total Collected 40
Unique <ArrowStringArray>
[                                                                                                                                              '🔥 TWS Wireless Earbuds with LED Display – ENC Noise Cancellation, HiFi Stereo, Bluetooth 5.3 Ear Buds |  woth Mic',
                                                                                                 'Lanteso True Wireless Earbuds, Bluetooth Earbuds with Mics Clear Call Touch Control Bluetooth Headphones with Bass Sound in Ear Earphones for Music,Home Office',
                                                                                                                                                      'Smart TWS Airbuds – Bluetooth 5.0 Wireless Earphones with AI Voice Translator, Touch Control, Charging Box',
                                                 'Air-podTWS 12 black Mini Bluetooth Wireless Headset_(True Wireless Stereo) Bluetooth Earbuds Bluetooth Handsfree  Dual Sound 

In [33]:
df.isna().sum()

Product_Name    0
Price           0
Brand           0
SKU             0
Category        0
Seller          0
Product_URL     0
dtype: int64

In [9]:
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
import os

driver = webdriver.Chrome()
review_data = []

# Top 10 mobile products ke reviews extract karna
for i, url in enumerate(product_links[30:40], start=1):
  # Protocol fix
  if url.startswith("//"):
    url = "https:" + url
  elif url.startswith("/"):
    url = "https://www.daraz.pk" + url

  if not url.startswith("http"):
    continue

  try:
    driver.get(url)
    time.sleep(3)

    # 1. Step-by-Step Scroll down to trigger Lazy Loading
    for scroll in range(1000, 3000, 700):
      driver.execute_script(f"window.scrollTo(0, {scroll});")
      time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # Product Name extract
    h1 = soup.find("h1")
    p_name = h1.get_text(strip=True) if h1 else "Mobile Product"

    # 2. Daraz Review Classes Search (Multiple Class Fallbacks)
    comment_blocks = (
        soup.find_all("div", class_="content")
        + soup.find_all("div", class_="item-content")
        + soup.find_all("div", class_="review-content-sl")
    )

    found_count = 0
    seen_texts = set()  # Duplicates se bachne ke liye

    for block in comment_blocks:
      text = block.get_text(strip=True)
      if text and text not in seen_texts and len(text) > 3:
        seen_texts.add(text)
        review_data.append({"Product_Name": p_name, "Review_Text": text})
        found_count += 1

    if found_count > 0:
      print(f"[{i}] Success: Extracted {found_count} reviews for: {p_name[:30]}")
    else:
      print(f"[{i}] No text reviews found for: {p_name[:30]}")

  except Exception as e:
    print(f"[{i}] Error loading page: {e}")

driver.quit()

# CSV Mein Save Karna
file_exists= os.path.exists('Mobile_Reviews.csv')
if review_data:
  df_reviews = pd.DataFrame(review_data)
  df_reviews.to_csv("Mobile_Reviews.csv",mode='a', index=False, encoding="utf-8",header= not file_exists)
  print(
      f"\nMubarak ho! Total {len(df_reviews)} reviews 'Mobile_Reviews.csv' mein"
      " save ho gaye hain."
  )
else:
  print("\nKoi review save nahi ho saka. Mobile links dubara verify karein.")

[1] Success: Extracted 7 reviews for: Air buds Transparent Air39 Blu
[2] Success: Extracted 7 reviews for: Air proo & Air 31 TWS  Airpood
[3] Success: Extracted 7 reviews for: Original M19 and M10 TWS Wirel
[4] Success: Extracted 7 reviews for: Trance Airbud 850 Hexa Mic wit
[5] Success: Extracted 7 reviews for: TWS Air_pods_ with Super Sound
[6] Success: Extracted 7 reviews for: Stereo GamingPro (2026) AI Tru
[7] Success: Extracted 7 reviews for: Air31 Earbuds Wireless Crystal
[8] Success: Extracted 7 reviews for: Air pod M04 Wireless  Bluetoot
[9] Success: Extracted 7 reviews for: EARBUDS Air buds Air 31 Wirele
[10] Success: Extracted 7 reviews for: Wireless Bluetooth5.2 Headphon

Mubarak ho! Total 70 reviews 'Mobile_Reviews.csv' mein save ho gaye hain.
